
# Pathway–Gene Topology Builder

This notebook constructs a pathway-informed gene interaction topology from Reactome pathway relationships and NCBI gene mappings.

## Workflow

1. Load Reactome pathway graph
2. Load Reactome–NCBI mappings
3. Load approved gene symbols
4. Filter to valid genes
5. Build pathway–gene associations
6. Generate pathway-pair → gene-pair relationships
7. Remove duplicates
8. Limit graph density
9. Export pathway-informed topology


In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from tqdm.auto import tqdm
import random
import os

tqdm.pandas()

# ==============================
# CONFIG
# ==============================

MAX_GENES_PER_PATHWAY = 500
RANDOM_TRUNCATION = False
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ==============================
# 0. Load OMICS GENE UNIVERSE
# ==============================

OMICS_FILE = (
    "data/processed/omics_genes.txt"
)

omics_genes = pd.read_table(
    OMICS_FILE,
    header=None
)[0].astype(str).str.upper().str.strip()


omics_genes = set(
    omics_genes
)


print(
    f"✅ Omics gene universe: {len(omics_genes)} genes"
)


# ==============================
# 1. Load driver / non-driver
# ==============================

drivers = pd.read_table(
    "data/processed/driver_genes.txt",
    # "../data/processed/763_driver_genes.txt",
    header=None
)[0].astype(str).str.upper().str.strip()


nondrivers = pd.read_table(
    "data/processed/non_driver_genes.txt",
    # "../data/processed/5263_non_driver_genes.txt",
    header=None
)[0].astype(str).str.upper().str.strip()


drivers = set(drivers) & omics_genes
nondrivers = set(nondrivers) & omics_genes


print(
    f"Drivers after omics filtering: {len(drivers)}"
)

print(
    f"Non-drivers after omics filtering: {len(nondrivers)}"
)



# ==============================
# 2. Load enrichment
# ==============================

enrichment = pd.read_csv(
    "../reactome_embedding/results/enrichment/enrichment_simple.csv"
)[
    [
        "stId",
        "p_value",
        "significance"
    ]
]


enrichment_dict = (
    enrichment
    .set_index("stId")
    .to_dict("index")
)



# ==============================
# 3. Load pathway genes
# ==============================

pathway_genes_raw = pd.read_csv(
    "../data/processed/pathways_mapped_all_genes.tsv",
    sep="\t"
)


pathway_to_genes = {}

removed_gene_count = 0


for _, row in pathway_genes_raw.iterrows():

    pid = row["PathwayID"]


    genes = [

        str(g).upper().strip()

        for g in row[1:]

        if pd.notna(g)
    ]


    # --------------------------
    # IMPORTANT:
    # keep only omics genes
    # --------------------------

    before = len(genes)


    genes = [

        g for g in genes

        if g in omics_genes
    ]


    removed_gene_count += (
        before - len(genes)
    )


    if RANDOM_TRUNCATION:

        random.shuffle(genes)


    genes = genes[
        :MAX_GENES_PER_PATHWAY
    ]


    if genes:

        pathway_to_genes[pid] = genes



print(
    f"✅ Loaded pathways: {len(pathway_to_genes)}"
)

print(
    f"🧹 Removed non-omics genes: {removed_gene_count}"
)



# ==============================
# 4. Load pathway relations
# ==============================

pathway_relations = pd.read_csv(
    "../data/processed/ReactomePathwaysRelation_filtered.tsv",
    sep="\t",
    header=None,
    names=[
        "PathwayA",
        "PathwayB"
    ]
)


valid_pathways = set(
    enrichment["stId"]
)


pathway_relations = pathway_relations[
    pathway_relations["PathwayA"].isin(valid_pathways)
    &
    pathway_relations["PathwayB"].isin(valid_pathways)
]


print(
    f"✅ Filtered relations: {len(pathway_relations)}"
)



# ==============================
# 5. Generate gene pairs
# ==============================

gene_pairs = []


for _, row in tqdm(
    pathway_relations.iterrows(),
    total=len(pathway_relations)
):

    pA = row["PathwayA"]
    pB = row["PathwayB"]


    genesA = pathway_to_genes.get(
        pA,
        []
    )

    genesB = pathway_to_genes.get(
        pB,
        []
    )


    if not genesA or not genesB:
        continue



    enrich_info = enrichment_dict.get(
        pA
    )


    if enrich_info is None:
        continue


    pval = enrich_info["p_value"]

    sig = enrich_info["significance"]



    for gA, gB in product(
        genesA,
        genesB
    ):

        # remove self loops

        if gA == gB:
            continue


        gene_pairs.append(
            (
                pA,
                gA,
                pB,
                gB,
                pval,
                sig
            )
        )



print(
    f"✅ Raw pairs: {len(gene_pairs)}"
)



# ==============================
# 6. DataFrame
# ==============================

df = pd.DataFrame(
    gene_pairs,
    columns=[
        "PathwayA",
        "Gene1",
        "PathwayB",
        "Gene2",
        "pvalue",
        "significance"
    ]
)



# ==============================
# 7. Remove duplicates
# ==============================


df.drop_duplicates(
    inplace=True
)


df["pair_key"] = df.apply(

    lambda x:

    tuple(sorted(
        [
            (
                x["PathwayA"],
                x["Gene1"]
            ),

            (
                x["PathwayB"],
                x["Gene2"]
            )
        ]
    )),

    axis=1
)



df.drop_duplicates(
    subset="pair_key",
    inplace=True
)


df.drop(
    columns="pair_key",
    inplace=True
)



print(
    f"✅ Unique pairs: {len(df)}"
)



# ==============================
# 8. Gene type
# ==============================

def get_gene_type(row):

    return int(

        row["Gene1"] in drivers

        or

        row["Gene2"] in drivers
    )



df["gene_type"] = (
    df.progress_apply(
        get_gene_type,
        axis=1
    )
)



# ==============================
# 9. Limit genes/pathway
# ==============================

def limit_per_pathway(
    data,
    max_genes=100
):

    output=[]


    for pid, group in data.groupby(
        "PathwayA"
    ):


        driver_df = group[
            group["gene_type"]==1
        ]


        other_df = group[
            group["gene_type"]!=1
        ]


        need = max_genes - len(driver_df)



        if need > 0:

            sampled = other_df.sample(

                n=min(
                    need,
                    len(other_df)
                ),

                random_state=RANDOM_STATE
            )


            final = pd.concat(
                [
                    driver_df,
                    sampled
                ]
            )

        else:

            final = driver_df.sample(
                max_genes,
                random_state=RANDOM_STATE
            )



        # TP53 preservation

        if "TP53" in omics_genes:

            tp53 = group[
                group["Gene1"]=="TP53"
            ]


            final = pd.concat(
                [
                    final,
                    tp53
                ]
            ).drop_duplicates()



        output.append(
            final
        )


    return pd.concat(
        output,
        ignore_index=True
    )



df_final = limit_per_pathway(
    df,
    MAX_GENES_PER_PATHWAY
)



# ==============================
# 10. Final safety filter
# ==============================


df_final = df_final[
    df_final["Gene1"].isin(omics_genes)
    &
    df_final["Gene2"].isin(omics_genes)
]



print(
    f"✅ Final omics-compatible pairs: {len(df_final)}"
)



# ==============================
# 11. Save
# ==============================


output_path = (

    f"data/processed/"
    f"pathway_informed_gene_gene_pairs_omics_{MAX_GENES_PER_PATHWAY}.csv"

)



df_final.to_csv(
    output_path,
    index=False
)



print(
    f"✅ Saved → {output_path}"
)

✅ Omics gene universe: 13627 genes
Drivers after omics filtering: 694
Non-drivers after omics filtering: 2957


/var/folders/z_/4_txl3tn61g8nprq0_s7tsbc0000gn/T/ipykernel_1469/2203222540.py:103: DtypeWarning: Columns (340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,56

✅ Loaded pathways: 2719
🧹 Removed non-omics genes: 0
✅ Filtered relations: 1904


  0%|          | 0/1904 [00:00<?, ?it/s]

✅ Raw pairs: 21768393


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105255790>>
Traceback (most recent call last):
  File "/Users/ericsali/miniforge3/envs/kg39/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105255790>>
Traceback (most recent call last):
  File "/Users/ericsali/miniforge3/envs/kg39/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [3]:
# ============================================================
# PATHWAY-INFORMED GENE-GENE PAIR GENERATION
# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------
# ✔ Progress bars for all major steps
# ✔ Only keeps genes existing in omics_genes.txt
# ✔ Driver preservation
# ✔ TP53 preservation
# ✔ Removes self loops
# ✔ Removes symmetric duplicates
# ✔ Limits genes per pathway
# ✔ Saves final training pairs
# ============================================================


import os
import random
import numpy as np
import pandas as pd

from itertools import product
from tqdm.auto import tqdm


tqdm.pandas()


# ============================================================
# CONFIG
# ============================================================

MAX_GENES_PER_PATHWAY = 500
RANDOM_TRUNCATION = False
RANDOM_STATE = 42


random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)



# ============================================================
# 0. LOAD OMICS GENE UNIVERSE
# ============================================================

print("\nLoading omics genes...")


omics_genes = pd.read_table(
    "data/processed/omics_genes.txt",
    header=None,
    names=["gene"]
)


omics_genes["gene"] = (
    omics_genes["gene"]
    .astype(str)
    .str.upper()
    .str.strip()
)


omics_genes = set(
    omics_genes["gene"]
)


print(
    f"✅ Omics genes: {len(omics_genes):,}"
)

# ============================================================
# 1. LOAD DRIVER / NON DRIVER
# ============================================================

print("\nLoading driver genes...")

drivers = pd.read_table(
    "../data/processed/796_drivers.txt",
    header=None,
    names=["gene"]
)

drivers = set(
    drivers["gene"]
    .astype(str)
    .str.upper()
    .str.strip()
)

nondrivers = pd.read_table(
    "../data/processed/2187_nondrivers.txt",
    header=None,
    names=["gene"]
)

nondrivers = set(
    nondrivers["gene"]
    .astype(str)
    .str.upper()
    .str.strip()
)

drivers &= omics_genes
nondrivers &= omics_genes

print(
    f"✅ Drivers kept: {len(drivers):,}"
)

print(
    f"✅ Non-drivers kept: {len(nondrivers):,}"
)



# ============================================================
# 2. ENRICHMENT
# ============================================================


print("\nLoading enrichment...")


enrichment = pd.read_csv(
    "../reactome_embedding/results/enrichment/enrichment_simple.csv"
)


enrichment = enrichment[
    [
        "stId",
        "p_value",
        "significance"
    ]
]


enrichment_dict = (

    enrichment
    .set_index("stId")
    .to_dict("index")
)



print(
    f"✅ Enriched pathways: {len(enrichment):,}"
)



# ============================================================
# 3. PATHWAY → GENES
# ============================================================


print("\nLoading pathway genes...")


pathway_raw = pd.read_csv(
    "../data/processed/pathways_mapped_all_genes.tsv",
    sep="\t"
)


pathway_to_genes = {}


removed = 0


for _, row in tqdm(
    pathway_raw.iterrows(),
    total=len(pathway_raw),
    desc="Filtering pathway genes"
):

    pid = row["PathwayID"]


    genes = [

        str(g)
        .upper()
        .strip()

        for g in row.iloc[1:]

        if pd.notna(g)
    ]


    before = len(genes)


    genes = [
        g for g in genes
        if g in omics_genes
    ]


    removed += (
        before - len(genes)
    )


    if RANDOM_TRUNCATION:

        random.shuffle(genes)


    genes = genes[
        :MAX_GENES_PER_PATHWAY
    ]


    if genes:

        pathway_to_genes[pid] = genes



print(
    f"✅ Pathways loaded: {len(pathway_to_genes):,}"
)

print(
    f"🧹 Removed non-omics genes: {removed:,}"
)



# ============================================================
# 4. PATHWAY RELATIONS
# ============================================================


print("\nLoading pathway relations...")


relations = pd.read_csv(
    "../data/processed/ReactomePathwaysRelation_filtered.tsv",
    sep="\t",
    header=None,
    names=[
        "PathwayA",
        "PathwayB"
    ]
)


valid = set(
    enrichment["stId"]
)


relations = relations[
    relations["PathwayA"].isin(valid)
    &
    relations["PathwayB"].isin(valid)
]


print(
    f"✅ Valid relations: {len(relations):,}"
)



# ============================================================
# 5. GENERATE GENE PAIRS
# ============================================================


print("\nGenerating gene pairs...")


pairs = []


for row in tqdm(
    relations.itertuples(index=False),
    total=len(relations),
    desc="Creating pairs"
):

    pA = row.PathwayA
    pB = row.PathwayB


    genesA = pathway_to_genes.get(
        pA,
        []
    )

    genesB = pathway_to_genes.get(
        pB,
        []
    )


    if not genesA or not genesB:
        continue


    info = enrichment_dict.get(pA)


    if info is None:
        continue



    for g1, g2 in product(
        genesA,
        genesB
    ):

        if g1 == g2:
            continue


        pairs.append(
            (
                pA,
                g1,
                pB,
                g2,
                info["p_value"],
                info["significance"]
            )
        )



print(
    f"✅ Raw pairs: {len(pairs):,}"
)



# ============================================================
# 6. DATAFRAME
# ============================================================


df = pd.DataFrame(

    pairs,

    columns=[
        "PathwayA",
        "Gene1",
        "PathwayB",
        "Gene2",
        "pvalue",
        "significance"
    ]
)



# ============================================================
# 7. REMOVE DUPLICATES
# ============================================================


print("\nRemoving duplicates...")


df.drop_duplicates(
    inplace=True
)



df["pair_key"] = (

    df[
        [
            "PathwayA",
            "Gene1",
            "PathwayB",
            "Gene2"
        ]
    ]
    .astype(str)
    .apply(
        lambda x:
        tuple(sorted(x)),
        axis=1
    )

)



df.drop_duplicates(
    subset="pair_key",
    inplace=True
)


df.drop(
    columns="pair_key",
    inplace=True
)


print(
    f"✅ Unique pairs: {len(df):,}"
)



# ============================================================
# 8. DRIVER LABEL
# ============================================================


print("\nAssigning driver labels...")


# df["gene_type"] = (

#     df["Gene1"].isin(drivers)
#     |
#     df["Gene2"].isin(drivers)

# ).astype(int)

# ============================================================
# 8. GENE TYPE + CONNECTED DRIVER GENE
# ============================================================

print("\nAssigning gene labels...")


def determine_gene_type(row):

    g1 = row["Gene1"]
    g2 = row["Gene2"]

    if g1 in drivers:
        return 1

    elif g1 in nondrivers:
        return 0

    elif g2 in drivers:
        return 2

    else:
        return 3


def get_connected_driver(row):

    g1 = row["Gene1"]
    g2 = row["Gene2"]

    if g1 not in drivers and g2 in drivers:
        return g2

    elif g1 in drivers:
        return g1

    return None


df["gene_type"] = (
    df.apply(
        determine_gene_type,
        axis=1
    )
)

df["connected_driver_gene"] = (
    df.apply(
        get_connected_driver,
        axis=1
    )
)

print(df["gene_type"].value_counts())

print(
    "\nConnected-driver edges:",
    df["connected_driver_gene"]
    .notna()
    .sum()
)

df = df[
    [
        "PathwayA",
        "Gene1",
        "PathwayB",
        "Gene2",
        "connected_driver_gene",
        "gene_type",
        "pvalue",
        "significance"
    ]
]

print("\nGene type counts")

for k, v in (
    df["gene_type"]
    .value_counts()
    .sort_index()
    .items()
):
    print(f"Type {k}: {v:,}")




# ============================================================
# 9. LIMIT PER PATHWAY
# ============================================================


def limit_per_pathway(
    group
):

    driver_rows = group[
        group.gene_type == 1
    ]


    other_rows = group[
        group.gene_type == 0
    ]


    needed = (
        MAX_GENES_PER_PATHWAY
        -
        len(driver_rows)
    )


    if needed > 0:

        sampled = other_rows.sample(
            min(
                needed,
                len(other_rows)
            ),
            random_state=RANDOM_STATE
        )


        final = pd.concat(
            [
                driver_rows,
                sampled
            ]
        )

    else:

        final = driver_rows.sample(
            MAX_GENES_PER_PATHWAY,
            random_state=RANDOM_STATE
        )



    # keep TP53

    tp53 = group[
        group.Gene1=="TP53"
    ]


    if len(tp53):

        final = pd.concat(
            [
                final,
                tp53
            ]
        ).drop_duplicates()



    return final



print("\nLimiting pathway size...")


df_final = (

    df
    .groupby(
        "PathwayA",
        group_keys=False
    )
    .progress_apply(
        limit_per_pathway
    )
    .reset_index(drop=True)

)



# ============================================================
# 10. FINAL SAFETY FILTER
# ============================================================


print("\nFinal omics filtering...")


df_final = df_final[
    df_final.Gene1.isin(omics_genes)
    &
    df_final.Gene2.isin(omics_genes)
]



print(
    f"✅ Final pairs: {len(df_final):,}"
)



# ============================================================
# 11. SAVE
# ============================================================


os.makedirs(
    "data/processed",
    exist_ok=True
)


output = (

    "data/processed/"
    f"pathway_informed_gene_gene_pairs_omics_"
    f"{MAX_GENES_PER_PATHWAY}.csv"

)



df_final.to_csv(
    output,
    index=False
)



print(
    "\n================================================"
)

print(
    f"🎯 SAVED: {output}"
)

print(
    "================================================"
)


Loading omics genes...
✅ Omics genes: 13,627

Loading driver genes...
✅ Drivers kept: 793
✅ Non-drivers kept: 2,186

Loading enrichment...
✅ Enriched pathways: 1,997

Loading pathway genes...


/var/folders/z_/4_txl3tn61g8nprq0_s7tsbc0000gn/T/ipykernel_65454/374083392.py:164: DtypeWarning: Columns (340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,56

Filtering pathway genes:   0%|          | 0/2719 [00:00<?, ?it/s]

✅ Pathways loaded: 2,719
🧹 Removed non-omics genes: 0

Loading pathway relations...
✅ Valid relations: 1,904

Generating gene pairs...


Creating pairs:   0%|          | 0/1904 [00:00<?, ?it/s]

✅ Raw pairs: 21,768,393

Removing duplicates...
✅ Unique pairs: 18,170,380

Assigning driver labels...

Assigning gene labels...
gene_type
3    12774013
1     2216540
2     1780676
0     1399151
Name: count, dtype: int64

Connected-driver edges: 4162832

Gene type counts
Type 0: 1,399,151
Type 1: 2,216,540
Type 2: 1,780,676
Type 3: 12,774,013

Limiting pathway size...


  0%|          | 0/617 [00:00<?, ?it/s]


Final omics filtering...
✅ Final pairs: 192,623

🎯 SAVED: data/processed/pathway_informed_gene_gene_pairs_omics_500.csv


## Configuration

In [4]:

import pandas as pd
import numpy as np

kg_file = 'data/kg/reactome_pathway_kg.csv'
ncbi_file = 'data/reactome/reactome_ncbi.csv'
gene_file = 'data/gene/gene_names.csv'
all_gene_file = 'data/gene_lists/all_genes.txt'

output_file = 'data/kg/pathway_gene_pairs.csv'

ea_result = {}
pval_threshold = 0.05

MAX_GENES_PER_PATHWAY = 30
MAX_EDGES_PER_GENE = 100


## Load Input Data

In [5]:

df_kg = pd.read_csv(kg_file)
df_ncbi = pd.read_csv(ncbi_file)
df_genes = pd.read_csv(gene_file, sep='\t')

with open(all_gene_file) as f:
    valid_genes = {
        line.strip()
        for line in f
    }

print('KG edges:', len(df_kg))
print('NCBI mappings:', len(df_ncbi))
print('Gene records:', len(df_genes))


FileNotFoundError: [Errno 2] No such file or directory: 'data/kg/reactome_pathway_kg.csv'

## Detect NCBI Identifier Column

In [ ]:

ncbi_col_candidates = [
    c for c in df_genes.columns
    if 'NCBI' in c
]

if not ncbi_col_candidates:
    raise ValueError('No NCBI ID column found')

ncbi_col = ncbi_col_candidates[0]

df_genes = (
    df_genes[[ncbi_col, 'Approved symbol']]
    .rename(columns={
        ncbi_col: 'ncbi_id',
        'Approved symbol': 'symbol'
    })
)

df_genes.head()


## Expand Multi-ID Gene Records

In [ ]:

df_genes['ncbi_id'] = (
    df_genes['ncbi_id']
    .astype(str)
    .str.split('|')
    .apply(lambda x: [i.strip() for i in x])
)

df_genes_exp = df_genes.explode('ncbi_id')

df_genes_exp = df_genes_exp[
    df_genes_exp['ncbi_id'].str.isnumeric()
]

print('Expanded mappings:', len(df_genes_exp))


## Build Pathway–Gene Mapping

In [ ]:

df_path_gene = pd.merge(
    df_ncbi,
    df_genes_exp,
    on='ncbi_id',
    how='inner'
)[['reactome_id', 'symbol']]

df_path_gene = df_path_gene[
    df_path_gene['symbol'].isin(valid_genes)
]

print('Filtered pathway-gene pairs:', len(df_path_gene))


## Assign Enrichment Statistics

In [ ]:

df_path_gene['pvalue'] = (
    df_path_gene['reactome_id']
    .map(lambda x: ea_result.get(x, 1.0))
)

df_path_gene['significance'] = (
    df_path_gene['pvalue']
    .apply(lambda x: 1 if x < pval_threshold else 0)
)

df_path_gene.head()


## Limit Genes Per Pathway

In [ ]:

df_path_gene = (
    df_path_gene
    .groupby('reactome_id')
    .head(MAX_GENES_PER_PATHWAY)
)

print('After pathway cap:', len(df_path_gene))


## Merge Source and Target Pathways

In [ ]:

df_src = pd.merge(
    df_kg,
    df_path_gene,
    left_on='x_id',
    right_on='reactome_id'
)

df_src = (
    df_src.rename(columns={
        'symbol': 'gene_x',
        'pvalue': 'pvalue_x'
    })
    .drop(columns=['reactome_id'])
)

df_full = pd.merge(
    df_src,
    df_path_gene,
    left_on='y_id',
    right_on='reactome_id'
)

df_full = (
    df_full.rename(columns={
        'symbol': 'gene_y',
        'pvalue': 'pvalue_y'
    })
    .drop(columns=['reactome_id'])
)

print('Merged records:', len(df_full))


## Generate Pathway-Informed Gene Topology

In [ ]:

df_full['pvalue'] = (
    df_full[['pvalue_x', 'pvalue_y']]
    .min(axis=1)
)

df_out = df_full[[
    'x_id',
    'gene_x',
    'y_id',
    'gene_y',
    'pvalue'
]].copy()

df_out.columns = [
    'PathwayA',
    'Gene1',
    'PathwayB',
    'Gene2',
    'pvalue'
]

df_out.head()


## Remove Duplicate Gene Pairs

In [ ]:

df_out = (
    df_out
    .sort_values('pvalue')
    .drop_duplicates(['Gene1', 'Gene2'])
)

print('Unique gene pairs:', len(df_out))


## Limit Connectivity

In [ ]:

df_out = (
    df_out
    .groupby('Gene1')
    .head(MAX_EDGES_PER_GENE)
)

print('Final edges:', len(df_out))


## Export Results

In [ ]:

df_out.to_csv(
    output_file,
    index=False
)

print(f'Saved {len(df_out)} edges')
print('Output:', output_file)
